In [1]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/D-alanine_opt_L-ala_cell-17O_magres.magres')

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [5]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-62.93377238 166.22451312 -80.56157078]
 [172.4697815   65.96113206  24.7350798 ]
 [  8.22591105 -25.69604004 -51.81500928]]

17O2 sigma:
 [[ -53.82014758  218.76780944   55.02898048]
 [ 185.81011053  115.33568175  -88.18842382]
 [  18.64191826  -51.17834508 -172.26916506]]

17O3 sigma:
 [[-62.93377238 166.22451312  80.56157078]
 [172.4697815   65.96113206 -24.7350798 ]
 [ -8.22591105  25.69604004 -51.81500928]]

17O4 sigma:
 [[ -62.93377238 -166.22451312   80.56157078]
 [-172.4697815    65.96113206   24.7350798 ]
 [  -8.22591105  -25.69604004  -51.81500928]]

17O5 sigma:
 [[ -62.93377238 -166.22451312  -80.56157078]
 [-172.4697815    65.96113206  -24.7350798 ]
 [   8.22591105   25.69604004  -51.81500928]]

17O6 sigma:
 [[ -53.82014758  218.76780944  -55.02898048]
 [ 185.81011053  115.33568175   88.18842382]
 [ -18.64191826   51.17834508 -172.26916506]]

17O7 sigma:
 [[ -53.82014758 -218.76780944  -55.02898048]
 [-185.81011053  115.33568175  -88.18842382]
 [ -18.64191826

In [6]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.3985376856896785

17O2 sigma:
 8.257931708019381

17O3 sigma:
 6.398537685689589

17O4 sigma:
 6.398537685689671

17O5 sigma:
 6.398537685689582

17O6 sigma:
 8.25793170801937

17O7 sigma:
 8.257931708019346

17O8 sigma:
 8.257931708019344



In [7]:
Q = -0.0256 #electric quadrupole moment for O17 in barn

CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + l = 1 + l = 2) Tensor from magres
CS_total[0,0] = -69.0859; CS_total[0,1] = 166.3401; CS_total[0,2] = -81.0135;
CS_total[1,0] = 172.0605; CS_total[1,1] = 69.1360; CS_total[1,2] = 24.6605;
CS_total[2,0] = 9.5405; CS_total[2,1] = -26.4671; CS_total[2,2] =   -52.9341;

Cs = np.zeros((3,3)) # CS symmetric (l = 0 + l = 2) 

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)
efg[0,0]= -0.1686; efg[0,1]= 0.1407; efg[0,2]= -0.7759;
efg[1,0]= efg[0,1]; efg[1,1]= 0.1091; efg[1,2]= 0.6093;
efg[2,0]= efg[0,2]; efg[2,1]= efg[1,2]; efg[2,2]= 0.0595;

# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 

V = efg*Q*234.9647

print(V)

print(Cs)

[[ 1.01414524 -0.84632405  4.66711323]
 [-0.84632405 -0.65624701 -3.66499819]
 [ 4.66711323 -3.66499819 -0.35789823]]
[[-69.0859 169.2003 -35.7365]
 [169.2003  69.136   -0.9033]
 [-35.7365  -0.9033 -52.9341]]


In [8]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 6.43465941 -0.83606708 -5.59859233] 

 Unsorted Eigenvectors:
 [[-0.63007444  0.62082859 -0.46645263]
 [ 0.4145751   0.7768494   0.47395411]
 [-0.65660771 -0.10524672  0.74685302]] 

Sorted Eigenvalues: 
 [-0.83606708 -5.59859233  6.43465941] 

Sorted Eigenvectors: 
 [[ 0.62082859 -0.46645263 -0.63007444]
 [ 0.7768494   0.47395411  0.4145751 ]
 [-0.10524672  0.74685302 -0.65660771]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 184.61310066 -189.0462867   -48.45081395] 

 Unsorted Eigenvectors:
 [[ 0.56123012  0.81786772 -0.12693757]
 [ 0.8230151  -0.53525246  0.19013402]
 [-0.08756083  0.21118048  0.97351729]] 

Sorted Eigenvalues: 
 [ -48.45081395 -189.0462867   184.61310066] 

Sorted Eigenvectors: 
 [[-0.12693757  0.81786772  0.56123012]
 [ 0.19013402 -0.53525246  0.8230151 ]
 [ 0.97351729  0.21118048 -0.08756083]] 



In [9]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.8360670816636966 -5.598592329440219 6.434659411103916
CSA Tensor Components δyy, δxx, δzz: 
 -48.450813954879315 -189.04628670364306 184.6131006585224


In [10]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        6.43466
etaq            0.740136
iso_cs (ppm)  -17.628
csa (ppm)     202.241
etas            0.695187


In [11]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.46645263  0.62082859 -0.63007444]
 [ 0.47395411  0.7768494   0.4145751 ]
 [ 0.74685302 -0.10524672 -0.65660771]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
-8.021317445938088 131.0416679775285 33.344035462222045 

Direction cosine csa: 

[[ 0.81786772 -0.12693757  0.56123012]
 [-0.53525246  0.19013402  0.8230151 ]
 [ 0.21118048  0.97351729 -0.08756083]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
77.7607269966613 95.02329875170486 -55.709157035962576 



In [12]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 28.744385948055985 chi: 87.4163492559477 xi: -86.3906692197498 



**Rotation of tensors Crystal--> Tenon Frame**

In [13]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[-69.0859 169.2003 -35.7365]
 [169.2003  69.136   -0.9033]
 [-35.7365  -0.9033 -52.9341]]
CSA Tensor in Tenon Frame: 
 [[ 107.42787001  -96.88257715 -115.79364724]
 [ -96.88257715  -63.56537001  -28.61802798]
 [-115.79364724  -28.61802798  -96.7465    ]]
Quad Tensor in Crystal Frame: 
 [[ 1.01414524 -0.84632405  4.66711323]
 [-0.84632405 -0.65624701 -3.66499819]
 [ 4.66711323 -3.66499819 -0.35789823]]
Quad Tensor in Tenon Frame: 
 [[-0.08561371 -1.24311864  3.02239706]
 [-1.24311864 -4.90962303  1.22953487]
 [ 3.02239706  1.22953487  4.99523674]]
